# SDH exp_024 — Exact-event EB 3-seed 제출 재현

학습은 자동 실행하지 않습니다. 아래 셀을 위에서 아래로 직접 실행하세요. seed별 셀이 오래 걸립니다.

In [ ]:
from pathlib import Path
import json
import sys
import numpy as np
import pandas as pd

REPO_ROOT = Path.cwd().resolve()
if not (REPO_ROOT / 'experiments' / 'SDH' / 'exp_024_exact_event_eb_reproduction').exists():
    REPO_ROOT = next((p for p in [REPO_ROOT, *REPO_ROOT.parents] if (p / 'experiments' / 'SDH' / 'exp_024_exact_event_eb_reproduction').exists()), None)
if REPO_ROOT is None:
    raise RuntimeError('exp24 worktree 루트에서 JupyterLab을 실행해 주세요.')
DATA_ROOT = next((p for p in [REPO_ROOT, *REPO_ROOT.parents] if (p / 'data' / 'raw' / 'train.csv').exists()), None)
if DATA_ROOT is None:
    raise RuntimeError('상위 경로에서 data/raw/train.csv를 찾지 못했습니다.')

EXP_DIR = REPO_ROOT / 'experiments' / 'SDH' / 'exp_024_exact_event_eb_reproduction'
RESULT_DIR = EXP_DIR / 'results'
RESULT_DIR.mkdir(parents=True, exist_ok=True)
if str(EXP_DIR) not in sys.path:
    sys.path.insert(0, str(EXP_DIR))

import exact_event_pipeline as exp
exp.ROOT_OVERRIDE = DATA_ROOT
exp.DATA_DIR_OVERRIDE = DATA_ROOT / 'data' / 'raw'
exp.OUTPUT_DIR_OVERRIDE = RESULT_DIR
print('REPO_ROOT:', REPO_ROOT)
print('DATA_ROOT:', DATA_ROOT)
print('pipeline:', Path(exp.__file__).resolve())

## 1. Train-only smoke test

이 셀은 `test.csv`를 읽지 않고 경로, 파서, NaN 처리, concat 금지를 확인합니다.

In [ ]:
exp.smoke_test()

## 2. 최종 추론 데이터 로드

아래부터 test를 읽습니다. vocabulary나 통계를 test에서 학습하지 않고 full-train fit 변환을 적용하는 데만 사용합니다.

In [ ]:
RAW = DATA_ROOT / 'data' / 'raw'
train = pd.read_csv(RAW / 'train.csv')
test = pd.read_csv(RAW / 'test.csv')
sample_submission = pd.read_csv(RAW / 'sample_submission.csv')
print('train:', train.shape, 'test:', test.shape, 'sample:', sample_submission.shape)

## 3. Seed 42

구조화 H0, gene×type EB, exact-event EB, LR, automatic specialist를 full train에서 학습합니다.

In [ ]:
prob_42, classes_42, audit_42 = exp.build_submission_probability(train, test, model_seed=42)
print('seed42:', prob_42.shape, 'feature_count:', audit_42['final_feature_count'])

## 4. Seed 777

In [ ]:
prob_777, classes_777, audit_777 = exp.build_submission_probability(train, test, model_seed=777)
print('seed777:', prob_777.shape, 'feature_count:', audit_777['final_feature_count'])

## 5. Seed 2024

In [ ]:
prob_2024, classes_2024, audit_2024 = exp.build_submission_probability(train, test, model_seed=2024)
print('seed2024:', prob_2024.shape, 'feature_count:', audit_2024['final_feature_count'])

## 6. 3-seed 동등 평균 및 제출 저장

세 seed의 클래스 순서를 확인하고 확률을 1/3씩 평균합니다.

In [ ]:
assert np.array_equal(classes_42, classes_777)
assert np.array_equal(classes_42, classes_2024)
probability = exp.average_seed_probabilities([prob_42, prob_777, prob_2024])
assert probability.shape == (len(test), len(classes_42))
assert np.isfinite(probability).all()
assert np.allclose(probability.sum(axis=1), 1.0, atol=1e-5)

submission = exp.make_submission_frame(sample_submission, test, probability, classes_42)
submission_path = RESULT_DIR / 'submission_h0_exact_event_eb_seed42_777_2024_bagged.csv'
submission.to_csv(submission_path, index=False)

audit = {
    'run_id': exp.RUN_ID + '-notebook-seed-bagging',
    'seeds': [42, 777, 2024],
    'seed_weights': [1/3, 1/3, 1/3],
    'raw_train_test_concat': False,
    'test_role': 'transform_and_predict_only',
    'leakage_check': all(a['leakage_check'] for a in [audit_42, audit_777, audit_2024]),
    'per_seed_audits': [audit_42, audit_777, audit_2024],
    'output_file': str(submission_path),
    'row_count': len(submission),
}
audit_path = submission_path.with_suffix('.audit.json')
audit_path.write_text(json.dumps(audit, ensure_ascii=False, indent=2), encoding='utf-8')
print('submission:', submission_path)
print('audit:', audit_path)
submission.head()

## 7. 최종 파일 재검사

In [ ]:
reloaded = pd.read_csv(submission_path)
assert reloaded.equals(submission)
assert reloaded['ID'].tolist() == test['ID'].tolist()
print('PASS:', reloaded.shape, 'duplicate ID:', reloaded['ID'].duplicated().sum())